# Configuration 3 -> 
## Openai-embedding-large


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/openai_large/all_chunks.csv")

In [3]:
df.head()

,id,text
0,/Users/midhunln/Documents/rag20march_with_eval...,Page 1 of 132 \n \n \nMASTER CIRCULAR \n \nSEB...
1,/Users/midhunln/Documents/rag20march_with_eval...,Securities and Exchange Board of India (Issue ...
2,/Users/midhunln/Documents/rag20march_with_eval...,Page 2 of 132 \n \n4. With the issuance of thi...
3,/Users/midhunln/Documents/rag20march_with_eval...,"suffered thereunder, any right, privilege, obl..."
4,/Users/midhunln/Documents/rag20march_with_eval...,regulations and bidding portal. \n7. All liste...


In [4]:
df["id"][0]

'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_0_0'

In [5]:
df.info

<bound method DataFrame.info of                                                      id  \
0     /Users/midhunln/Documents/rag20march_with_eval...   
1     /Users/midhunln/Documents/rag20march_with_eval...   
2     /Users/midhunln/Documents/rag20march_with_eval...   
3     /Users/midhunln/Documents/rag20march_with_eval...   
4     /Users/midhunln/Documents/rag20march_with_eval...   
...                                                 ...   
1103  /Users/midhunln/Documents/rag20march_with_eval...   
1104  /Users/midhunln/Documents/rag20march_with_eval...   
1105  /Users/midhunln/Documents/rag20march_with_eval...   
1106  /Users/midhunln/Documents/rag20march_with_eval...   
1107  /Users/midhunln/Documents/rag20march_with_eval...   

                                                   text  
0     Page 1 of 132 \n \n \nMASTER CIRCULAR \n \nSEB...  
1     Securities and Exchange Board of India (Issue ...  
2     Page 2 of 132 \n \n4. With the issuance of thi...  
3     suffered thereunder, 

In [6]:
df_sampled = df.sample(frac = 0.1)

In [7]:
df_sampled.head()

,id,text
997,/Users/midhunln/Documents/rag20march_with_eval...,Page 242 of 261 \nANNEXURE 22 \n \nFORMAT FOR ...
705,/Users/midhunln/Documents/rag20march_with_eval...,• Conclude on the appropriateness of the Board...
804,/Users/midhunln/Documents/rag20march_with_eval...,Name of authority Brief of the case Correctiv...
207,/Users/midhunln/Documents/rag20march_with_eval...,Page 87 of 132 \n \n \nb. ASBA: Bank Account N...
342,/Users/midhunln/Documents/rag20march_with_eval...,"/ year ended March 31, 2017, the formats were ..."


In [8]:
!pip install langchain-ollama


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model = "llama3")

/Users/midhunln/Documents/rag20march_with_eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model="llama3")

def llm_call(text):
    prompt = f"""
You are a query generator. Based on the text below, generate a concise search query
that a user would likely use to find this information. The output must only have the query and nothing else.
It is very important that the output is only the query and nothing else.

Text:
{text}
"""
    response = chat.invoke([HumanMessage(content=prompt)])
    return response.content.strip()


# Apply row-wise correctly
df_sampled["query"] = df_sampled["text"].apply(llm_call)

In [11]:
df_sampled["query"].iloc[2]

'"public policy positions advocated by entity"'

In [12]:
df_sampled.to_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/openai_large/chunk_to_query_mapping.csv", index = False)

In [13]:
df_sampled.head()

,id,text,query
997,/Users/midhunln/Documents/rag20march_with_eval...,Page 242 of 261 \nANNEXURE 22 \n \nFORMAT FOR ...,"""voting results submission format"""
705,/Users/midhunln/Documents/rag20march_with_eval...,• Conclude on the appropriateness of the Board...,"""auditor's report going concern"""
804,/Users/midhunln/Documents/rag20march_with_eval...,Name of authority Brief of the case Correctiv...,"""public policy positions advocated by entity"""
207,/Users/midhunln/Documents/rag20march_with_eval...,Page 87 of 132 \n \n \nb. ASBA: Bank Account N...,"""Asba application process"""
342,/Users/midhunln/Documents/rag20march_with_eval...,"/ year ended March 31, 2017, the formats were ...","""Companies Act 2013 financial results publicat..."


# Construct the qrels dataset for rankx

In [14]:
qrels = {}

for _, row in df_sampled.iterrows():
    query = row["query"]
    doc_id = row["id"]

    if query not in qrels:
        qrels[query] = {}

    qrels[query][doc_id] = 1  # relevance score

In [15]:
qrels

{'"voting results submission format"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_241_47': 1},
 '"auditor\'s report going concern"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_127_5': 1},
 '"public policy positions advocated by entity"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_172_4': 1},
 '"As

# Start retriever


In [16]:
!pip install langchain-pinecone


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [17]:
import sys
sys.path.append('/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval')

from dotenv import load_dotenv
load_dotenv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/ingestion.env")

from Repositories.pinecone_repository import PineconeRepository

In [18]:
from Repositories.pinecone_repository import PineconeConfig
from src.openai_embedding import OpenAIEmbedding
from src.sparse_embedding import SentenceTransformerSparseEmbedding

In [19]:
config = PineconeConfig()

dense_embedding_strategy = OpenAIEmbedding(config)

sparse_embedding_strategy = SentenceTransformerSparseEmbedding(config)


Loading weights: 100%|██████████| 204/204 [00:00<00:00, 31608.35it/s]
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
import os

In [21]:
pinecone_repository = PineconeRepository(
            api_key=os.getenv("PINECONE_API_KEY"),
            environment=os.getenv("PINECONE_ENVIRONMENT", "us-east-1-aws"),
            dense_embedding_strategy=dense_embedding_strategy,
            sparse_embedding_strategy=sparse_embedding_strategy,
            pinecone_config=config
        )

In [22]:
df_sampled["relevant_docs"] = df_sampled["query"].apply(pinecone_repository.query_vector_store_for_rankx)

In [23]:
df_sampled.to_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/openai_large/chunk_query_relevant_docs.csv", index = False)

In [24]:
df_sampled.head()

,id,text,query,relevant_docs
997,/Users/midhunln/Documents/rag20march_with_eval...,Page 242 of 261 \nANNEXURE 22 \n \nFORMAT FOR ...,"""voting results submission format""",[{'id': '/Users/midhunln/Documents/rag20march_...
705,/Users/midhunln/Documents/rag20march_with_eval...,• Conclude on the appropriateness of the Board...,"""auditor's report going concern""",[{'id': '/Users/midhunln/Documents/rag20march_...
804,/Users/midhunln/Documents/rag20march_with_eval...,Name of authority Brief of the case Correctiv...,"""public policy positions advocated by entity""",[{'id': '/Users/midhunln/Documents/rag20march_...
207,/Users/midhunln/Documents/rag20march_with_eval...,Page 87 of 132 \n \n \nb. ASBA: Bank Account N...,"""Asba application process""",[{'id': '/Users/midhunln/Documents/rag20march_...
342,/Users/midhunln/Documents/rag20march_with_eval...,"/ year ended March 31, 2017, the formats were ...","""Companies Act 2013 financial results publicat...",[{'id': '/Users/midhunln/Documents/rag20march_...


In [25]:
df_sampled["relevant_docs"].iloc[0]

[{'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_37_17',
  'score': 25.9270592},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_241_47',
  'score': 20.3109837},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_60_35',
  'score': 19.9712582},
 {'id': '/Users/midhunln/Documents/rag20march_with_

In [26]:
qrels_dict = {}

for i, row in df_sampled.iterrows():
    qid = str(i)
    true_doc = row["id"]   # single doc id string
    qrels_dict[qid] = {true_doc: 1}

In [27]:
run_dict = {}

for i, row in df_sampled.iterrows():
    qid = str(i)
    
    run_dict[qid] = {
        doc["id"]: float(doc["score"])
        for doc in row["relevant_docs"]
    }

In [28]:
from ranx import Qrels, Run, evaluate

qrels = Qrels(qrels_dict)
run = Run(run_dict)

metrics = [
    "mrr",
    "ndcg@10",
    "recall@5",
    "recall@10",
    "precision@5"
]

results = evaluate(qrels, run, metrics)
print(results)

{'mrr': np.float64(0.601176176176176), 'ndcg@10': np.float64(0.6547493420526447), 'recall@5': np.float64(0.7567567567567568), 'recall@10': np.float64(0.8198198198198198), 'precision@5': np.float64(0.1513513513513513)}


In [29]:
final_result = pd.Series(results)


In [32]:
result = pd.Series(results)

In [33]:
result

mrr            0.601176
ndcg@10        0.654749
recall@5       0.756757
recall@10      0.819820
precision@5    0.151351
dtype: float64